# 📘 Aula 08a: Visão Computacional com Imagens Reais – Dogs vs. Cats, Data Augmentation e Transfer Learning com VGG16

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula08a_dogs_vs_cats.ipynb)

Nas aulas anteriores, trabalhamos com o dataset sintético e controlado do MNIST ($28 \times 28$ em escala de cinza). Agora, ingressamos no mundo da **Visão Computacional Aplicada a Fotografias Reais**: o famoso desafio do Kaggle **Dogs vs. Cats**.

Trabalharemos com fotografias coloridas de alta resolução em formatos variados de cães e gatos. O grande desafio pedagógico deste laboratório é o cenário de **dados limitados (*small data regime*)**: treinaremos nossos modelos utilizando apenas **2.000 imagens** (1.000 gatos e 1.000 cães). Você verá na prática a evolução completa da engenharia de visão computacional moderna:
1. Uma **CNN construída do zero** que sofre com severo *overfitting*.
2. O poder da técnica de **Aumento de Dados (Data Augmentation)** e **Dropout** para combater a memorização.
3. A revolução do **Aprendizado por Transferência (Transfer Learning)** utilizando uma rede pré-treinada na ImageNet (**VGG16**), atingindo mais de **97% de acurácia** com um punhado de fotos!

---

### 🎯 Objetivo da Aula
1. Configurar o download e descompactação de datasets diretamente da API do Kaggle.
2. Estruturar diretórios em disco (`train`, `validation`, `test`) usando `pathlib` e realizar streaming eficiente de imagens com `image_dataset_from_directory`.
3. Construir e treinar uma CNN profunda do zero com Keras (`Conv2D` e `MaxPooling2D` empilhados) monitorando com `ModelCheckpoint`.
4. Diagnosticar visualmente o *overfitting* agressivo provocado pela escassez de amostras de treino.
5. Implementar camadas de **Data Augmentation** (`RandomFlip`, `RandomRotation`, `RandomZoom`) e visualizar suas transformações geométricas no Matplotlib.
6. Utilizar a rede pré-treinada **VGG16** (treinada em 1.4 milhão de imagens da ImageNet), congelando suas camadas convolucionais (`conv_base.trainable = False`) para extração de características profundas.
7. Integrar **EarlyStopping** e **Dropout** para alcançar o estado da arte com generalização impecável.

---

### 💡 A Intenção Pedagógica do Professor
* **A Progressão Didática Perfeita**: Em vez de começar direto com redes gigantes pré-treinadas, o professor primeiro deixa o aluno ver a CNN sofrer com o dataset pequeno, compreendendo *por que* o overfitting ocorre e *como* cada técnica introduzida resolve uma deficiência real.
* **Streaming de Imagens**: Demonstrar que em problemas do mundo real não carregamos todas as fotos na memória RAM de uma vez; usamos iteradores em disco (`image_dataset_from_directory`).
* **Transfer Learning Prático**: Ensinar o conceito mais valioso do profissional de Deep Learning moderno: como reaproveitar filtros visuais treinados por supercomputadores para resolver seu problema específico com poucos dados e poucos minutos de GPU.

---

### 📚 Glossário da Aula
* **Data Augmentation (Aumento de Dados)**: Geração de novas amostras de treino aplicando transformações aleatórias geometricamente válidas (espelhamentos, rotações, zooms) nas imagens existentes.
* **Transfer Learning (Aprendizado por Transferência)**: Técnica de reaproveitar o conhecimento (pesos e filtros) aprendido por uma rede neural em um dataset gigante para uma tarefa secundária com poucos dados.
* **VGG16**: Arquitetura convolucional de 16 camadas desenvolvida pela Universidade de Oxford, treinada na base ImageNet (1.000 classes).
* **Feature Extraction (Extração de Características)**: Utilizar a base convolucional de uma rede pré-treinada sem alterar seus pesos (`trainable = False`) e treinar apenas um novo classificador no topo.
* **ModelCheckpoint**: Callback do Keras que monitora a perda de validação e salva no disco apenas o melhor modelo obtido durante todas as épocas.

---

### 🧭 Trilha da Aula
1. [Autenticação e Download do Dataset via Kaggle API](#1-autenticação-e-download-do-dataset-via-kaggle-api)
2. [Organização dos Subsets em Disco com pathlib](#2-organização-dos-subsets-de-imagens-em-disco)
3. [Carregamento e Streaming de Imagens com image_dataset_from_directory](#3-streaming-de-imagens-com-image_dataset_from_directory)
4. [Fase 1: CNN Construída do Zero](#4-fase-1-cnn-construída-do-zero-sem-regularização)
5. [Diagnóstico de Overfitting e Avaliação no Teste](#5-diagnóstico-de-overfitting-e-avaliação)
6. [Fase 2: Combate ao Overfitting com Data Augmentation e Dropout](#6-fase-2-combate-ao-overfitting-com-data-augmentation-e-dropout)
7. [Visualização Gráfica das Variações de Augmentation](#7-visualização-gráfica-do-data-augmentation)
8. [Fase 3: Transfer Learning com VGG16 (ImageNet)](#8-fase-3-transfer-learning-com-vgg16-pré-treinada)
9. [Fase 4: Transfer Learning com Data Augmentation, Dropout e EarlyStopping](#9-fase-4-transfer-learning-completo-com-earlystopping)


## 1. Autenticação e Configuração da API do Kaggle

**O que estamos fazendo**: Criar a pasta `.kaggle` e configurar o token de acesso com permissões restritas (chmod 600).

**Por que fazer assim**: Permite que o script faça download seguro e automático do acervo Dogs vs Cats sem intervenção manual.

> [!NOTE]
> O comando `chmod 600` define que apenas o usuário proprietário tem permissão de leitura/escrita no token de acesso.



In [ ]:
# Cria o diretório de credenciais do Kaggle e salva o token de acesso com permissão segura
mkdir -p ~/.kaggle && echo KGAT_1b5ba89aa89a15c38329f43058d64f34 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token


## 2. Instalação e Atualização da CLI do Kaggle

**O que estamos fazendo**: Instalar o pacote `kaggle` oficial do Python via pip.

**Por que fazer assim**: Garante a versão mais recente da interface de linha de comando para download de competições.



In [ ]:
# Atualiza o cliente de linha de comando oficial do Kaggle
!pip install --upgrade kaggle


## 3. Download da Competição Dogs vs Cats Redux

**O que estamos fazendo**: Baixar o arquivo zip oficial da competição Dogs vs Cats Redux kernels edition.

**Por que fazer assim**: A competição contém 25.000 imagens fotográficas coloridas de cães e gatos.



In [ ]:
# Baixa os arquivos zip da competição Dogs vs Cats Redux
!kaggle competitions download -c dogs-vs-cats-redux-kernels-edition


## 4. Descompactação do Arquivo Principal

**O que estamos fazendo**: Descompactar o arquivo principal baixado da competição.

**Por que fazer assim**: Extrai os arquivos individuais, incluindo `train.zip`.



In [ ]:
# Descompacta silenciosamente o pacote zip principal
!unzip -q dogs-vs-cats-redux-kernels-edition.zip


## 5. Descompactação das Imagens de Treino

**O que estamos fazendo**: Descompactar as 25.000 imagens da pasta `train.zip`.

**Por que fazer assim**: Gera a pasta `train` com todos os arquivos no formato `cat.0.jpg`, `dog.0.jpg`, etc.



In [ ]:
# Descompacta silenciosamente a pasta contendo as fotos de treino
!unzip -q train.zip


## 6. Configuração dos Caminhos com pathlib

**O que estamos fazendo**: Definir os diretórios da pasta de origem original e da nova pasta de trabalho reduzida `cats_vs_dogs_small`.

**Por que fazer assim**: O módulo `pathlib.Path` oferece manipulação de caminhos orientada a objetos moderna e independente de sistema operacional.



In [ ]:
# Importa módulos padrão para operações no sistema de arquivos
import os, shutil, pathlib

# Diretório com todas as 25.000 imagens descompactadas
original_dir = pathlib.Path('train')
# Novo diretório onde criaremos o dataset reduzido balanceado
new_base_dir = pathlib.Path('cats_vs_dogs_small')


## 7. Organização dos Subsets em Disco (Small Data Regime)

**O que estamos fazendo**: Criar as subpastas `train`, `validation` e `test` contendo pastas separadas para `cat` e `dog` com subsets controlados de imagens.

**Por que fazer assim**: Esta estrutura hierárquica (`diretório/classe/imagem.jpg`) é o padrão universal esperado por utilitários modernos de Deep Learning como `image_dataset_from_directory`.

> [!NOTE]
> Distribuição das amostras:
- **Treino**: 1.000 gatos + 1.000 cães (2.000 total)
- **Validação**: 500 gatos + 500 cães (1.000 total)
- **Teste**: 1.000 gatos + 1.000 cães (2.000 total)



In [ ]:
def make_subset(subset_name, start_index, end_index):
    """
    Cria subconjuntos organizados em pastas para cada categoria (cat e dog).
    """
    for category in ('cat', 'dog'):
        # Cria o diretório: cats_vs_dogs_small / subset_name / category
        dir = new_base_dir / subset_name / category
        os.makedirs(dir, exist_ok=True)
        # Gera a lista de nomes dos arquivos para a fatia definida
        fnames = [f"{category}.{i}.jpg" for i in range(start_index, end_index)]
        # Copia cada imagem para o destino correspondente
        for fname in fnames:
            shutil.copyfile(src=original_dir / fname, dst=dir / fname)

# Cria a partição de treino (1.000 imagens de cada classe)
make_subset('train', start_index=0, end_index=1000)
# Cria a partição de validação (500 imagens de cada classe)
make_subset('validation', start_index=1000, end_index=1500)
# Cria a partição de teste (1.000 imagens de cada classe)
make_subset('test', start_index=1500, end_index=2500)
print('Subsets de imagens organizados com sucesso!')


## 8. Arquitetura da CNN do Zero (Modelo Baseline)

**O que estamos fazendo**: Montar uma rede convolucional sequencial profunda recebendo imagens coloridas de $180 \times 180 \times 3$, com `Rescaling(1./255)`, 5 blocos convolucionais e saída sigmoide para classificação binária.

**Por que fazer assim**: Para imagens coloridas $180 \times 180$, precisamos de várias camadas de pooling para reduzir progressivamente a dimensão espacial até que ela possa alimentar uma camada densa sem explosão de parâmetros.

> [!NOTE]
> A camada `layers.Rescaling(1./255)` normaliza os valores dos pixels de $[0, 255]$ para $[0.0, 1.0]$ diretamente como uma camada da rede!



In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# Camada de entrada para imagens coloridas RGB de 180x180 pixels
inputs = keras.Input(shape=(180, 180, 3))
# Normaliza a escala de cores de [0, 255] para [0, 1]
x = layers.Rescaling(1./255)(inputs)
# Bloco Convolucional 1: 32 filtros 3x3 + MaxPooling 2x2
x = layers.Conv2D(filters=32, kernel_size=3, activation='relu')(x)
x = layers.MaxPooling2D(pool_size=2)(x)
# Bloco Convolucional 2: 64 filtros 3x3 + MaxPooling 2x2
x = layers.Conv2D(filters=64, kernel_size=3, activation='relu')(x)
x = layers.MaxPooling2D(pool_size=2)(x)
# Bloco Convolucional 3: 128 filtros 3x3 + MaxPooling 2x2
x = layers.Conv2D(filters=128, kernel_size=3, activation='relu')(x)
x = layers.MaxPooling2D(pool_size=2)(x)
# Bloco Convolucional 4: 256 filtros 3x3 + MaxPooling 2x2
x = layers.Conv2D(filters=256, kernel_size=3, activation='relu')(x)
x = layers.MaxPooling2D(pool_size=2)(x)
# Bloco Convolucional 5: 256 filtros 3x3
x = layers.Conv2D(filters=256, kernel_size=3, activation='relu')(x)
# Achatamento para vetor 1D
x = layers.Flatten()(x)
# Saída binária: 1 neurônio ativado por Sigmoide (probabilidade entre 0 e 1)
outputs = layers.Dense(1, activation='sigmoid')(x)

# Cria e compila o modelo
model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(loss='binary_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])
model.summary()


## 9. Streaming de Imagens com image_dataset_from_directory

**O que estamos fazendo**: Instanciar os geradores de dados para treino, validação e teste com redimensionamento automático para $180 \times 180$ e lotes de 32 imagens.

**Por que fazer assim**: Permite carregar e processar as imagens sob demanda durante o treino, consumindo memória RAM mínima.

> [!TIP]
> A função infere os rótulos automaticamente a partir dos nomes das subpastas (`cat` -> 0, `dog` -> 1)!



In [ ]:
from tensorflow.keras.utils import image_dataset_from_directory

# Cria o dataset iterável de treino
train_ds = image_dataset_from_directory(
    new_base_dir / 'train',
    image_size=(180, 180),
    batch_size=32)

# Cria o dataset iterável de validação
validation_ds = image_dataset_from_directory(
    new_base_dir / 'validation',
    image_size=(180, 180),
    batch_size=32)

# Cria o dataset iterável de teste
test_ds = image_dataset_from_directory(
    new_base_dir / 'test',
    image_size=(180, 180),
    batch_size=32)


## 10. Inspeção de Lotes Gerados

**O que estamos fazendo**: Extrair um minibatch do dataset de treino e inspecionar o formato dos tensores de imagens e rótulos.

**Por que fazer assim**: Confirma que o lote contém 32 imagens de formato $(32, 180, 180, 3)$ e 32 rótulos binários.



In [ ]:
# Itera sobre o primeiro lote gerado pelo dataset
for data_batch, labels_batch in train_ds:
    print('Formato do lote de imagens:', data_batch.shape)
    print('Formato do lote de rótulos:', labels_batch.shape)
    break


## 11. Configuração do Callback ModelCheckpoint

**O que estamos fazendo**: Configurar o `ModelCheckpoint` para salvar automaticamente no arquivo `convnet_from_scratch.keras` apenas o modelo que obtiver a menor perda de validação (`save_best_only=True`).

**Por que fazer assim**: Garante que, mesmo que a rede sofra overfitting nas épocas finais, o modelo salvo no disco seja a versão com a melhor generalização.



In [ ]:
# Configura o callback para persistir o melhor checkpoint com base na perda de validação
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='convnet_from_scratch.keras',
        save_best_only=True,
        monitor='val_loss')
]


## 12. Treinamento da CNN do Zero por 30 Épocas

**O que estamos fazendo**: Treinar a rede convolucional básica nas 2.000 imagens de treino com validação nas 1.000 de validação.

**Por que fazer assim**: Vamos monitorar a evolução do aprendizado para diagnosticar a capacidade da rede em um dataset pequeno.



In [ ]:
# Executa o treinamento e registra o histórico de métricas
history = model.fit(
    train_ds,
    epochs=30,
    validation_data=validation_ds,
    callbacks=callbacks
)


## 13. Avaliação no Conjunto de Teste

**O que estamos fazendo**: Carregar o melhor checkpoint salvo no disco e avaliar a acurácia no conjunto de teste independente.

**Por que fazer assim**: Fornece a medida objetiva do desempenho da CNN básica.



In [ ]:
# Carrega o melhor modelo salvo pelo callback
test_model = keras.models.load_model('convnet_from_scratch.keras')
# Avalia nos dados de teste
test_loss, test_acc = test_model.evaluate(test_ds)
print(f'Acurácia no Teste (CNN do Zero): {test_acc:.3f}')


## 14. Gráfico da Acurácia de Treino vs Validação

**O que estamos fazendo**: Plotar as curvas de acurácia de treinamento e validação ao longo das 30 épocas.

**Por que fazer assim**: Evidencia a clássica divergência de overfitting: o treino caminha para 100%, mas a validação trava em torno de ~70%.



In [ ]:
import matplotlib.pyplot as plt

accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
epochs = range(1, len(accuracy) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, accuracy, 'bo', label='Acurácia no Treino')
plt.plot(epochs, val_accuracy, 'b', label='Acurácia na Validação')
plt.title('Acurácia de Treino e Validação (CNN do Zero)')
plt.xlabel('Épocas')
plt.ylabel('Acurácia')
plt.legend()
plt.grid(True)
plt.show()


## 15. Gráfico da Perda (Loss) de Treino vs Validação

**O que estamos fazendo**: Plotar a perda no treino e na validação.

**Por que fazer assim**: A perda de validação atinge o mínimo por volta da 10ª época e sobe drasticamente a partir daí, provando o sobreajuste severo provocado pela escassez de dados.

> [!WARNING]
> Esse gráfico é a prova cabal de que redes neurais profundas memorizam datasets pequenos facilmente!



In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 5))
plt.plot(epochs, loss, 'bo', label='Perda no Treino')
plt.plot(epochs, val_loss, 'b', label='Perda na Validação')
plt.title('Perda de Treinamento e Validação (CNN do Zero)')
plt.xlabel('Épocas')
plt.ylabel('Custo (Loss)')
plt.legend()
plt.grid(True)
plt.show()


## 16. Fase 2: Criando o Módulo de Data Augmentation

**O que estamos fazendo**: Definir um pipeline de aumento de dados sequencial com `RandomFlip('horizontal')`, `RandomRotation(0.1)` e `RandomZoom(0.2)`.

**Por que fazer assim**: Ao aplicar pequenas transformações aleatórias a cada passagem, o modelo nunca vê a mesma imagem duas vezes exatamente igual, quebrando a capacidade de memorização espúria.

> [!NOTE]
> Camadas de Data Augmentation só ficam ativas durante o treinamento (`training=True`). Durante a validação e teste, elas se comportam como funções de identidade neutras!



In [ ]:
# Bloco sequencial de aumento de dados do Keras
data_augmentation = keras.Sequential([
    # Espelhamento horizontal aleatório (gatos virados para a esquerda/direita continuam sendo gatos)
    layers.RandomFlip('horizontal'),
    # Rotação aleatória de até 10% do círculo trigonométrico
    layers.RandomRotation(0.1),
    # Zoom aleatório de até 20%
    layers.RandomZoom(0.2)
])


## 17. Visualização Gráfica do Data Augmentation

**O que estamos fazendo**: Pegar uma única imagem do lote de treino e exibir 9 variações geradas dinamicamente pelo pipeline de aumento de dados.

**Por que fazer assim**: Comprova visualmente como a mesma fotografia original é rotacionada, espelhada e aproximada de forma contínua.



In [ ]:
# Visualiza 9 variações aumentadas da mesma fotografia
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
    for i in range(9):
        augmented_images = data_augmentation(images)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented_images[0].numpy().astype('uint8'))
        plt.title(f'Variação Augmentada {i+1}')
        plt.axis('off')
plt.tight_layout()
plt.show()


## 18. CNN com Data Augmentation e Dropout

**O que estamos fazendo**: Reconstruir a rede inserindo `data_augmentation` na entrada e `layers.Dropout(0.5)` antes da camada densa de saída.

**Por que fazer assim**: O Dropout zera aleatoriamente 50% das conexões durante o treino, forçando a rede a aprender representações distribuídas redundantes e robustas.

> [!TIP]
> A combinação de Data Augmentation + Dropout é a dupla de defesa mais poderosa contra o overfitting em visão computacional!



In [ ]:
# Arquitetura aprimorada com Augmentation e Dropout
inputs = keras.Input(shape=(180, 180, 3))
# 1. Aplica aumento de dados na entrada
x = data_augmentation(inputs)
# 2. Normaliza para [0, 1]
x = layers.Rescaling(1./255)(x)
# 3. Convoluções e poolings
x = layers.Conv2D(filters=32, kernel_size=3, activation='relu')(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=64, kernel_size=3, activation='relu')(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation='relu')(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation='relu')(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation='relu')(x)
x = layers.Flatten()(x)
# 4. Dropout de 50% antes da camada final
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(loss='binary_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])


## 19. Treinamento com Regularização por 100 Épocas

**O que estamos fazendo**: Treinar a rede regularizada por 100 épocas salvando o checkpoint `convnet_from_scratch_with_augmentation.keras`.

**Por que fazer assim**: Como o modelo não sofre mais com overfitting precoce, ele pode ser treinado por muito mais tempo, atingindo acurácia muito superior.



In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='convnet_from_scratch_with_augmentation.keras',
        save_best_only=True,
        monitor='val_loss')
]

history = model.fit(
    train_ds,
    epochs=100,
    validation_data=validation_ds,
    callbacks=callbacks
)


## 20. Avaliação do Modelo Regularizado

**O que estamos fazendo**: Carregar e avaliar o modelo no conjunto de teste independente.

**Por que fazer assim**: A taxa de acerto salta de ~70% para a faixa de **~82% a 84%**, comprovando o sucesso das técnicas de regularização.



In [ ]:
# Avalia o modelo treinado com aumento de dados no teste
test_model = keras.models.load_model('convnet_from_scratch_with_augmentation.keras')
test_loss, test_acc = test_model.evaluate(test_ds)
print(f'Acurácia no Teste (Com Augmentation e Dropout): {test_acc:.3f}')


## 21. Curvas de Aprendizado com Regularização

**O que estamos fazendo**: Plotar os gráficos de perda e acurácia das 100 épocas.

**Por que fazer assim**: As curvas de treino e validação agora caminham muito mais próximas, sem a catastrófica divergência anterior.



In [ ]:
accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
epochs = range(1, len(accuracy) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, accuracy, 'bo', label='Treino')
plt.plot(epochs, val_accuracy, 'b', label='Validação')
plt.title('Acurácia (Augmentation + Dropout)')
plt.legend()
plt.grid(True)

loss = history.history['loss']
val_loss = history.history['val_loss']
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, 'bo', label='Treino')
plt.plot(epochs, val_loss, 'b', label='Validação')
plt.title('Perda (Augmentation + Dropout)')
plt.legend()
plt.grid(True)
plt.show()


## 22. Fase 3: Carregando a Base Convolucional da VGG16 (ImageNet)

**O que estamos fazendo**: Instanciar a arquitetura VGG16 com pesos pré-treinados na base `imagenet`, excluindo a camada densa original (`include_top=False`), e congelando seus pesos (`conv_base.trainable = False`).

**Por que fazer assim**: A ImageNet possui 1.4 milhão de imagens e 1.000 classes. A base convolucional já sabe extrair bordas, pelagens, olhos e focinhos perfeitamente. Congelar os pesos preserva esse conhecimento.

> [!NOTE]
> `conv_base.trainable = False` congela os 14.7 milhões de parâmetros da VGG16, impedindo que o gradiente os destrua durante o aprendizado!



In [ ]:
# Carrega a base convolucional pré-treinada VGG16
conv_base = keras.applications.vgg16.VGG16(
    weights='imagenet',          # Pesos aprendidos na base ImageNet
    include_top=False,           # Exclui a camada de classificação densa original de 1.000 classes
    input_shape=(180, 180, 3)    # Dimensão das nossas fotos
)

# Congela os pesos para que atuem puramente como extratores fixos de características
conv_base.trainable = False
conv_base.summary()


## 23. Acoplando Data Augmentation e o Novo Classificador à VGG16

**O que estamos fazendo**: Montar a rede encadeando: Entrada -> Data Augmentation -> `preprocess_input` da VGG16 -> `conv_base` (congelada) -> `Flatten` -> `Dense(256)` -> `Dense(1, sigmoid)`.

**Por que fazer assim**: A função `vgg16.preprocess_input` converte canais RGB para BGR e centraliza os pixels na média da ImageNet.

> [!TIP]
> Apenas a camada Dense de 256 neurônios e o neurônio de saída terão seus pesos ajustados durante o treino!



In [ ]:
inputs = keras.Input(shape=(180, 180, 3))
# Aumento de dados dinâmico
x = data_augmentation(inputs)
# Pré-processamento específico esperado pela VGG16
x = keras.applications.vgg16.preprocess_input(x)
# Passa pela base convolucional profunda congelada
x = conv_base(x)
x = layers.Flatten()(x)
# Classificador denso intermediário
x = layers.Dense(256)(x)
# Saída binária
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, outputs)
model.compile(loss='binary_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])
model.summary()


## 24. Treinamento com Transfer Learning por 20 Épocas

**O que estamos fazendo**: Treinar o classificador por 20 épocas salvando o checkpoint `transfer_learning_with_data_augmentation.keras`.

**Por que fazer assim**: Como a base convolucional já está treinada, a convergência é extremamente veloz.



In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='transfer_learning_with_data_augmentation.keras',
        save_best_only=True,
        monitor='val_loss')
]

history = model.fit(
    train_ds,
    epochs=20,
    validation_data=validation_ds,
    callbacks=callbacks
)


## 25. Avaliação do Transfer Learning Inicial

**O que estamos fazendo**: Carregar e avaliar o modelo no teste independente.

**Por que fazer assim**: A acurácia dá um salto impressionante para **~96% a 97%** com apenas 2.000 imagens de treino!



In [ ]:
# Avalia no teste o modelo de Transfer Learning
test_model = keras.models.load_model('transfer_learning_with_data_augmentation.keras')
test_loss, test_acc = test_model.evaluate(test_ds)
print(f'Acurácia no Teste (Transfer Learning): {test_acc:.3f}')


## 26. Curvas de Desempenho do Transfer Learning

**O que estamos fazendo**: Plotar acurácia e perda do modelo com base congelada.

**Por que fazer assim**: A acurácia de validação atinge patamares altíssimos desde as primeiras épocas.



In [ ]:
accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
epochs = range(1, len(accuracy) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, accuracy, 'bo', label='Treino')
plt.plot(epochs, val_accuracy, 'b', label='Validação')
plt.title('Acurácia Transfer Learning')
plt.legend()
plt.grid(True)

loss = history.history['loss']
val_loss = history.history['val_loss']
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, 'bo', label='Treino')
plt.plot(epochs, val_loss, 'b', label='Validação')
plt.title('Perda Transfer Learning')
plt.legend()
plt.grid(True)
plt.show()


## 27. Fase 4: Transfer Learning Refinado com Dropout

**O que estamos fazendo**: Adicionar `layers.Dropout(0.5)` entre a camada `Dense(256)` e a camada de saída.

**Por que fazer assim**: Mesmo com Transfer Learning, a camada densa de 256 neurônios possui muitos parâmetros que podem se sobreajustar; o Dropout garante robustez adicional.



In [ ]:
inputs = keras.Input(shape=(180, 180, 3))
x = data_augmentation(inputs)
x = keras.applications.vgg16.preprocess_input(x)
x = conv_base(x)
x = layers.Flatten()(x)
x = layers.Dense(256)(x)
# Dropout de 50% para proteger a camada densa contra sobreajuste
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, outputs)
model.compile(loss='binary_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])


## 28. Treinamento com EarlyStopping e ModelCheckpoint

**O que estamos fazendo**: Treinar por até 50 épocas utilizando tanto o `ModelCheckpoint` quanto o `EarlyStopping(patience=3)`.

**Por que fazer assim**: O `EarlyStopping` interrompe o treinamento automaticamente se a perda de validação não melhorar por 3 épocas consecutivas, restaurando os melhores pesos obtidos.

> [!TIP]
> Essa é a combinação profissional padrão recomendada para produção!



In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='transfer_learning_with_data_augmentation_and_dropout.keras',
        save_best_only=True,
        monitor='val_loss'),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True)
]

history = model.fit(
    train_ds,
    epochs=50,
    validation_data=validation_ds,
    callbacks=callbacks
)


## 29. Avaliação Final do Modelo Estado da Arte

**O que estamos fazendo**: Avaliar o modelo final no conjunto de teste independente.

**Por que fazer assim**: Consolida a performance definitiva com generalização exemplar.



In [ ]:
# Avalia no teste o modelo final com Transfer Learning, Augmentation, Dropout e EarlyStopping
test_model = keras.models.load_model('transfer_learning_with_data_augmentation_and_dropout.keras')
test_loss, test_acc = test_model.evaluate(test_ds)
print(f'Acurácia Final no Teste: {test_acc * 100:.2f}%')


## 30. Visualização das Curvas Finais de Aprendizado

**O que estamos fazendo**: Plotar os gráficos de perda e acurácia do modelo final refinado.

**Por que fazer assim**: Observe o comportamento suave e controlado tanto na perda quanto na acurácia.



In [ ]:
accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
epochs = range(1, len(accuracy) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, accuracy, 'bo', label='Treino')
plt.plot(epochs, val_accuracy, 'b', label='Validação')
plt.title('Acurácia Final Refinada')
plt.legend()
plt.grid(True)

loss = history.history['loss']
val_loss = history.history['val_loss']
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, 'bo', label='Treino')
plt.plot(epochs, val_loss, 'b', label='Validação')
plt.title('Perda Final Refinada')
plt.legend()
plt.grid(True)
plt.show()


---

## 🎓 Resumo da Aula & O Que Você Aprendeu

Nesta aula, realizamos a jornada completa do desenvolvimento prático de Visão Computacional moderna em cenários de poucos dados (*small data regime*):

1. **Estruturação de Dados com `pathlib`**: Aprendemos a organizar fotografias em subpastas hierárquicas e fazer streaming de imagens em lote via `image_dataset_from_directory`.
2. **A Armadilha da Escassez de Dados**: Vimos que uma CNN clássica treinada do zero com apenas 2.000 imagens sofre *overfitting* massivo, estacionando em ~70% de acurácia.
3. **Data Augmentation & Dropout**: Introduzimos transformações geométricas aleatórias (`RandomFlip`, `RandomRotation`, `RandomZoom`) e desativação aleatória de neurônios, elevando o teste para ~84%.
4. **Revolução do Transfer Learning com VGG16**: Congelamos a base convolucional pré-treinada na ImageNet (`conv_base.trainable = False`), reaproveitando filtros visuais de alto nível e saltando a acurácia para **mais de 97%** com poucas épocas!
5. **Boas Práticas de Produção**: Empregamos os callbacks `ModelCheckpoint` e `EarlyStopping` para salvar o modelo ótimo e prevenir desperdício de processamento computacional.

---

## 👉 Próximo Passo
Na **Aula 09a**, mudaremos de domínio: sairemos do espaço bidimensional das imagens para explorar o eixo contínuo do tempo com **Previsão de Séries Temporais (*Time Series Forecasting*)**, comparando modelos Densos, Convoluções 1D e Redes Recorrentes (SimpleRNN e LSTM)!
